# Dev Tutorial 1: FEEC

This tutorial is a first introduction to the use of finite element exterior calculus (FEEC) in Struphy. It is not meant to be a comprehensive introduction to FEEC, but rather a practical guide to how to use it in Struphy. For a more comprehensive introduction to FEEC, see [the numerics section of the Struphy doc](https://struphy-hub.github.io/struphy/sections/subsections/numerics-geomFE.html) and the references therein.

## Boundary conditions: basics

Our first aim is to project a simple function into the H1 and L2 spaces. We will do this for Derham objects with different boundary conditions and plot the results. Here is our function:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fun = lambda e1, e2, e3: np.cos(np.pi/2 * e1) 
e1 = np.linspace(0, 1, 100)
e2 = 0.5
e3 = 0.5
plt.plot(e1, fun(e1, e2, e3))
plt.xlabel('e1')
plt.show()

First, create a Derham object with 16 elements in the first direction:

In [ ]:
from struphy.feec.psydac_derham import Derham

derham = Derham(Nel=(16, 1, 1))

By default, the spline degrees are 1 in each direction:

In [ ]:
derham.p

By default, the boundary conditions are periodic in each direction (no boundary conditions):

In [ ]:
derham.bcs

Let us apply the geometric projector `P0` into the discrete H1 space. The result will be of type `StencilVector`:

In [ ]:
vec = derham.P0(fun) 
print(type(vec))

The `StencilVector` is Psydac's (and thus Struphy's) fundamental data structure for FE coefficients. It has many attributes and methods attached to it, which will be discussed in more detail in a future tutorial. For now, we will just look at what the `.shape` method returns, to make clear that this is not just a normal numpy array:

In [ ]:
print(vec.shape)
print(vec[:].shape)

More on `StencilVectors` later. One convenient method for developers is `.toarray()`, which converts the `StencilVector` into a normal numpy array. This is useful for plotting and debugging:

In [ ]:
vec.toarray()

The `StencilVector` thus holds FE coefficients. To get a function that we can evaluate at arbitrary points, we can use the `create_spline_function` method of the Derham object. This creates a callable object that evaluates the FE function at given points:

In [ ]:
fun_h = derham.create_spline_function(name="hello world",
                                      space_id="H1",
                                      coeffs=vec,
                                      )
plt.plot(e1, fun(e1, e2, e3), label="exact")
plt.plot(e1, fun_h(e1, e2, e3, squeeze_out=True), "r--", label="H1 (periodic)")
plt.xlabel('e1')
plt.legend()
plt.show()

Ooops - something went wrong here! Indeed, the periodic boundary conditions are not compatible with the function we are trying to project. Let us change the boundary conditions to be non-periodic in the first direction, and periodic in the other two:

In [ ]:
derham = Derham(Nel=(16, 1, 1), bcs=(("hom_dirichlet", "hom_dirichlet"), None, None))
vec = derham.P0(fun)
fun_h = derham.create_spline_function(name="hello world",
                                      space_id="H1",
                                      coeffs=vec,
                                      )
plt.plot(e1, fun(e1, e2, e3), label="exact")
plt.plot(e1, fun_h(e1, e2, e3, squeeze_out=True), "r--", label="H1 (hom_dirichlet)")
plt.xlabel('e1')
plt.legend()
plt.show()

Wrong again! The homogeneous Dirichlet conditions do not respect the non-zero boundary condition on the left. Let us change the boundary condition to be free boundary on the left:

In [ ]:
derham = Derham(Nel=(16, 1, 1), bcs=(("free", "hom_dirichlet"), None, None))
vec = derham.P0(fun)
fun_h = derham.create_spline_function(name="hello world",
                                      space_id="H1",
                                      coeffs=vec,
                                      )
plt.plot(e1, fun(e1, e2, e3), label="exact")
plt.plot(e1, fun_h(e1, e2, e3, squeeze_out=True), "r--", label="H1 (free and hom_dirichlet)")
plt.xlabel('e1')
plt.legend()
plt.show()

Voilà - that does the job. So far we projected into the discrete H1 space. For comparison, let us also project into the discrete L2 space, with the wrong boundary conditions for illustration purposes:

In [ ]:
derham = Derham(Nel=(16, 1, 1), bcs=(("hom_dirichlet", "hom_dirichlet"), None, None))

vec0 = derham.P0(fun)
vec3 = derham.P3(fun)

fun0_h = derham.create_spline_function(name="hello world H1",
                                      space_id="H1",
                                      coeffs=vec0,
                                      )
fun3_h = derham.create_spline_function(name="hello world L2",
                                      space_id="L2",
                                      coeffs=vec3,
                                      )
plt.plot(e1, fun(e1, e2, e3), label="exact")
plt.plot(e1, fun0_h(e1, e2, e3, squeeze_out=True), "r--", label="H1 (hom_dirichlet)")
plt.plot(e1, fun3_h(e1, e2, e3, squeeze_out=True), "g--", label="L2 (hom_dirichlet)")
plt.xlabel('e1')
plt.legend()
plt.show()

Two things to note here:

1. The discrete L2 space is less regular (piece-wise constant) than the discrete H1 space (piece-wise linear for degree `p=1`).

2. Even though we set the boundary conditions for the whole Derham complex, they are not actually applied to the L2 space, since this space does not have any boundary conditions. This is a common source of confusion for developers new to FEEC, so it is worth pointing out explicitly. It becomes clear when thinking of the 1d Derham sequence: applying the derivative to a function in H1_0 (with hom. Dirichlet conditions) gives a function in L2. But the derivative at the boundary can have any value, and is not forced to be zero. This thought applies to any component of a spline function that is a `D-spline`: they cannot be set to a specific value. 



## Neumann boundary conditions



## Lifting boundary conditions